In [ ]:
from google.colab import files

uploaded = files.upload()

Saving gdp_per_capita.csv to gdp_per_capita (1).csv
Saving GeoLite2-City.mmdb to GeoLite2-City (1).mmdb
Saving global_pop_2020_CN_1km_R2025A_UA_v1.tif to global_pop_2020_CN_1km_R2025A_UA_v1 (1).tif
Saving internet_users.csv to internet_users (1).csv
Saving population_density.csv to population_density (1).csv
Saving updated_cybersecurity_attacks.csv to updated_cybersecurity_attacks (1).csv


In [ ]:
!pip install geoip2
!pip install rasterio
!pip install pyasn
!pip install xgboost
!pip install lightgbm
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import geopandas as gpd
import rasterio
import pyasn
import ipaddress

from folium.plugins import HeatMap

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.utils import resample

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from libpysal.weights import KNN
from esda.moran import Moran
from esda.getisord import G_Local

import geoip2.database

In [ ]:
df = pd.read_csv(
    'updated_cybersecurity_attacks.csv',
    encoding='latin1'
)

In [ ]:

# FEATURE ENGINEERING

df.fillna("Unknown", inplace=True)

df.drop_duplicates(inplace=True)

df['Timestamp'] = pd.to_datetime(
    df['Timestamp'],
    errors='coerce'
)

df['hour'] = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['month'] = df['Timestamp'].dt.month

text_cols = [
    'Protocol',
    'Traffic Type',
    'Attack Type',
    'Severity Level'
]

df[text_cols] = df[text_cols].apply(
    lambda x: x.str.lower()
)

def ip_to_int(ip):
    try:
        return int(ipaddress.ip_address(ip))
    except:
        return np.nan

df['source_ip_numeric'] = (
    df['Source IP Address']
    .apply(ip_to_int)
)



/tmp/ipykernel_20196/3509071154.py:7: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Timestamp'] = pd.to_datetime(


In [ ]:
reader = geoip2.database.Reader(
    'GeoLite2-City.mmdb'
)

def get_location(ip):

    try:

        response = reader.city(ip)

        return pd.Series([

            response.location.latitude,
            response.location.longitude,
            response.country.name,
            response.city.name

        ])

    except:

        return pd.Series([
            None,
            None,
            None,
            None
        ])

df[['Latitude',
    'Longitude',
    'Country',
    'City']] = (
        df['Source IP Address']
        .apply(get_location)
)

reader.close()

df['Latitude'] = pd.to_numeric(
    df['Latitude'],
    errors='coerce'
)

df['Longitude'] = pd.to_numeric(
    df['Longitude'],
    errors='coerce'
)

df = df[
    (df['Latitude'].between(-90, 90))
]

df = df[
    (df['Longitude'].between(-180, 180))
]


In [ ]:
internet_df = pd.read_csv(
    'internet_users.csv'
)

gdp_df = pd.read_csv(
    'gdp_per_capita.csv'
)

population_df = pd.read_csv(
    'population_density.csv'
)



In [ ]:
internet_df = internet_df[
    ['Country Name', '2023 [YR2023]']
]

internet_df.columns = [
    'Country',
    'Internet_Users'
]

gdp_df = gdp_df[
    ['Country Name', '2023 [YR2023]']
]

gdp_df.columns = [
    'Country',
    'GDP_Per_Capita'
]

population_df = population_df[
    ['Country Name', '2023 [YR2023]']
]

population_df.columns = [
    'Country',
    'Population_Density'
]

df = df.merge(
    internet_df,
    on='Country',
    how='left'
)

df = df.merge(
    gdp_df,
    on='Country',
    how='left'
)

df = df.merge(
    population_df,
    on='Country',
    how='left'
)

In [ ]:
worldpop = rasterio.open(
    'global_pop_2020_CN_1km_R2025A_UA_v1.tif'
)

# READ BAND ONLY ONCE
population_band = worldpop.read(1)

# FAST FUNCTION
def get_population_density(lat, lon):

    try:

        row, col = worldpop.index(
            lon,
            lat
        )

        return population_band[row, col]

    except:

        return np.nan

# APPLY
df['local_population_density'] = [

    get_population_density(lat, lon)

    for lat, lon in zip(
        df['Latitude'],
        df['Longitude']
    )
]

In [ ]:
# ASN / ISP ENRICHMENT

asndb = pyasn.pyasn('ipasn.dat')

def get_asn(ip):

    try:
        return asndb.lookup(ip)[0]

    except:
        return np.nan

df['ASN'] = (
    df['Source IP Address']
    .apply(get_asn)
)

# CREATE HIGH RISK LABEL

df['High_Risk'] = (
    df['Severity Level']
    .apply(
        lambda x:
            1 if x in ['high', 'critical']
            else 0
    )
)


# ASN FEATURES

df['asn_attack_count'] = (
    df.groupby('ASN')['Attack Type']
      .transform('count')
)

df['asn_risk_score'] = (
    df.groupby('ASN')['High_Risk']
      .transform('mean')
)

# COUNTRY FEATURES

df['country_attack_count'] = (
    df.groupby('Country')['Attack Type']
      .transform('count')
)

In [ ]:
# Heatmap
heat_data = df[['Latitude','Longitude']].values.tolist()
heat_map = folium.Map(location=map_center, zoom_start=2)
HeatMap(heat_data).add_to(heat_map)

heat_map

In [ ]:

# DBSCAN CLUSTERING

coords = df[['Latitude','Longitude']].dropna()
coords_scaled = StandardScaler().fit_transform(coords)

db = DBSCAN(eps=0.3, min_samples=10)
df.loc[coords.index, 'Cluster'] = db.fit_predict(coords_scaled)

In [ ]:

# MACHINE LEARNING

df['High_Risk'] = df['Severity Level'].apply(
    lambda x: 1 if x in ['high','critical'] else 0
)

cat_cols = ['Protocol','Traffic Type','Attack Type','Country','City']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

features = ['Latitude','Longitude','Protocol','Traffic Type',
            'Attack Type','Anomaly Scores','day_of_week']

# Balance dataset
df_majority = df[df.High_Risk == 0]
df_minority = df[df.High_Risk == 1]

df_minority_upsampled = resample(df_minority,
                                replace=True,
                                n_samples=len(df_majority),
                                random_state=42)

df_balanced = pd.concat([df_majority, df_minority_upsampled])

X = df_balanced[features]
y = df_balanced['High_Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=2
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:,1]
preds = (probs > 0.6).astype(int)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

In [ ]:

# CONFUSION MATRIX

cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:

# SPATIAL ANALYSIS (MORAN + Gi*)

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Longitude, df.Latitude))

w = KNN.from_dataframe(gdf, k=5)
w.transform = 'r'

moran = Moran(gdf['Anomaly Scores'], w)
print("Moran’s I:", moran.I)

g_local = G_Local(gdf['Anomaly Scores'], w)
gdf['GiZScore'] = g_local.Zs

In [ ]:

# HOTSPOT VISUALIZATION

plt.figure(figsize=(10,6))
plt.scatter(gdf['Longitude'], gdf['Latitude'],
            c=gdf['GiZScore'], cmap='coolwarm')
plt.colorbar(label='Gi* Z-Score')
plt.title("Hotspot Detection")
plt.show()

In [ ]:

# DASHBOARD PLOTS

sns.scatterplot(x='Longitude', y='Latitude', hue='High_Risk', data=df)
plt.title("High-Risk Distribution")
plt.show()

sns.countplot(x='day_of_week', hue='High_Risk', data=df)
plt.title("High-Risk Over Time")
plt.show()

In [ ]:
#FEATURE ENGINEERING


# Attack frequency per location
df['location_attack_count'] = df.groupby(['Latitude','Longitude'])['Cluster'].transform('count')

# Country-level attack frequency
df['country_attack_count'] = df.groupby('Country')['Cluster'].transform('count')

# Hour risk encoding
df['hour_risk'] = df.groupby('hour')['High_Risk'].transform('mean')

# Protocol risk encoding
df['protocol_risk'] = df.groupby('Protocol')['High_Risk'].transform('mean')

# Anomaly × Time interaction
df['anomaly_time'] = df['Anomaly Scores'] * df['hour']

# Cluster risk
df['cluster_risk'] = df.groupby('Cluster')['High_Risk'].transform('mean')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
print("RF Accuracy:", accuracy_score(y_test, rf_preds))
print(classification_report(y_test, rf_preds))

In [ ]:
xgboost = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=2
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:,1]
preds = (probs > 0.6).astype(int)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

In [ ]:
!pip install lightgbm
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(n_estimators=150)
lgb.fit(X_train, y_train)

lgb_preds = lgb.predict(X_test)
print("Accuracy:", accuracy_score(y_test, lgb_preds))
print(classification_report(y_test, lgb_preds))

In [ ]:
!pip install catboost
from catboost import CatBoostClassifier

cat = CatBoostClassifier(verbose=0)
cat.fit(X_train, y_train)

cat_preds = cat.predict(X_test)
print("Accuracy:", accuracy_score(y_test, cat_preds))
print(classification_report(y_test, cat_preds))

In [ ]:
final_preds = (
    preds + rf_preds + lgb_preds + cat_preds
) / 4

final_preds = (final_preds > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, final_preds))
print(classification_report(y_test, final_preds))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model_dl = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_dl.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_dl.fit(X_train, y_train, epochs=10, batch_size=32)

dl_preds = (model_dl.predict(X_test) > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, dl_preds))
print(classification_report(y_test, dl_preds))